# 04 · Görsel model (Adım 8, kutu 3) — ChangeOS pilot
Kaggle: **GPU T4 x2**, internet açık, dataset `melihkotman/disasterm3-turkiye` ve (GitHub planı bitene kadar) kod için `kod` dataset'i ekli.
Protokol ve karar kuralları: `docs/adim8_pilot_kaggle.md` (sonuçlardan ÖNCE yazıldı). **Test bölmesine ve Osmaniye katlamasına dokunulmaz.**

In [ ]:
# 1) Kurulum (ever-beta 0.5.6 sabit!)
!pip install -q timm "ever-beta==0.5.6" albumentations tifffile einops "datasets[vision]"
!pip install -q --no-deps "git+https://github.com/Z-Zheng/pytorch-change-models.git"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Kodu al: A) GitHub (plan yenilendiyse)   B) 'kod' dataset'i (aşağıda yolu uyarla)
import os, subprocess, sys, glob
REPO = "/kaggle/working/deprem-vlm-rag"
if not os.path.exists(REPO):
    cand = glob.glob("/kaggle/input/**/deprem-vlm-rag", recursive=True)
    if cand:
        subprocess.run(["cp", "-r", cand[0], REPO], check=True)
    else:
        # GitHub özel repo: Secrets'ta GITHUB_TOKEN 'Attached' olmalı
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        subprocess.run(["git", "clone", f"https://{tok}@github.com/MelihKotman/Deprem-VLM-RAG.git", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/MelihKotman/Deprem-VLM-RAG.git"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
!python -m src.config && python -m src.dm3_verify

In [ ]:
# 3) Birim testleri (model/GPU gerekmez; 1 dk): ölçüt ve oylama kodu doğru mu?
!python -m src.selftest_eval

## Aşama 0 — hat testi (≈10–15 dk): süre, bellek, veri bekleme, kayıp düşüyor mu
`--pilot`: 64 eğitim karosu, 3 epoch, 40 karoluk değerlendirme. Değerlendirme **Malatya katlaması** (Adıyaman+Osmaniye ile eğitilir). Çıktıdaki `EXTRAPOLATION...` tahmindir, ölçüm değildir.

In [ ]:
!python -m src.train_changeos --scenario S1 --holdout Malatya --pilot --pilot-tiles 64 --epochs 3 \
    --post-shift 16 --cache-dir /kaggle/working/cache --workers 4 --out /kaggle/working/runs/p0_hat_testi

In [ ]:
!cat /kaggle/working/runs/p0_hat_testi/timing.json
!tail -n 3 /kaggle/working/runs/p0_hat_testi/log.jsonl

## Aşama 1 — sonrası öteleme aralığı (0 / 16 / 32 px), diğer her şey sabit
Kota/süre Aşama 0'daki ölçüme göre `--epochs` ile ayarlanır (Aşama 0 ölçümüyle 20'ye sabitlendi). Üç koşu aynı seed (42). Her biri `--max-minutes` ile korunur; kesilirse aynı komut + `--resume`.

In [ ]:
for R in (0, 16, 32):
    !python -m src.train_changeos --scenario S1 --holdout Malatya --epochs 20 --post-shift {R} \
        --cache-dir /kaggle/working/cache --workers 4 --max-minutes 60 --out /kaggle/working/runs/p1_shift{R}

In [ ]:
# (İSTEĞE BAĞLI) iki GPU'yu aynı anda kullan: öteleme 0 ve 16 paralel, sonra 32. Kota sayacının GPU başına mı oturum başına mı
# işlediği DOĞRULANMADI; ilk kullanımda Kaggle'ın kota göstergesinden kontrol et. Veri bekleme için --workers 2.
import subprocess
def launch(gpu, R):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(f"python -m src.train_changeos --scenario S1 --holdout Malatya --epochs 20 --post-shift {R} "
        f"--cache-dir /kaggle/working/cache --workers 2 --max-minutes 90 --out /kaggle/working/runs/p1_shift{R} "
        f"> /kaggle/working/runs_p1_shift{R}.log 2>&1", shell=True, env=env)
ps = [launch(0, 0), launch(1, 16)]
[p.wait() for p in ps]
launch(0, 32).wait()

## Aşama 1b — TANI (Aşama 2'den ÖNCE; yeniden eğitim yok, ~15 dk)
Aşama 1'de katı = tol5 = tol15 ve hasar-F1 ≤ 0,08 çıktı → ölçüt ayırt edici değil. Önce neden: oylama sınıf önseli mi, eğitimde öğrenememe mi, kayma mı? Protokol: `docs/adim8_pilot_kaggle.md` §6.

In [ ]:
# (a) Mevcut koşulların ayrıntısı: karışım matrisi, kademe kırılımı, son eğitim kayıpları
import json
for name in ("p1_shift0", "p1_shift16", "p1_shift32"):
    d = f"/kaggle/working/runs/{name}"
    M = json.load(open(f"{d}/metrics.json"))["by"]
    A = M["ALL"]
    print("==", name)
    for lvl in ("strict", "tol5", "tol15"):
        r = A[lvl]; print(f"  {lvl}: karışım(gerçek x tahmin yok/sağlam/hasarlı/yıkık) {r['confusion_true_x_pred0123']} FP {r['n_false_alarm']} kaçırılan {r['n_missed']}")
    print("  piksel:", {k: round(v, 3) for k, v in A["pixel"].items()})
    for k, v in M.items():
        if k.startswith("tier |"):
            print(f"  {k}: karo {v['n_tiles']} bileşen {v['strict']['n_gt_components']} katı hasarF1 {v['strict']['damage_f1']:.3f} tol15 {v['tol15']['damage_f1']:.3f}")
    print("  son kayıp:", open(f"{d}/log.jsonl").read().strip().splitlines()[-1])

In [ ]:
# (b) Eğitim İÇİ kontrol: model eğitim karolarında hasarı öğrenmiş mi? (genelleme değil, yalnız yetersiz öğrenme testi)
!python -m src.train_changeos --scenario S1 --holdout Malatya --eval-only --eval-in-sample --ckpt /kaggle/working/runs/p1_shift0/model_final.pt \
    --cache-dir /kaggle/working/cache --out /kaggle/working/runs/d_shift0_insample

In [ ]:
# (c) Oylama sınıf önseli: aynı ağırlıklarla üç oylama (yeniden eğitim yok). Ağırlıklar yalnız eğitim bina sayılarından türetilir.
for V in ("none", "sqrt_inv", "inv"):
    !python -m src.train_changeos --scenario S1 --holdout Malatya --eval-only --vote {V} --ckpt /kaggle/working/runs/p1_shift0/model_final.pt \
        --cache-dir /kaggle/working/cache --out /kaggle/working/runs/d_shift0_vote_{V}
!python -m src.summarize_runs /kaggle/working/runs

## Aşama 2 — sınıf dengesi (en iyi öteleme sabit): hasar odaklı örnekleme ± CE sınıf ağırlığı
`--dam-p 0.5` (varsayılan) ile `--dam-p 0.5 --cls-weights 1,1,3,3` karşılaştırılır; ayrıca `--dam-p 0.0` (tek örnekleme, referans).

In [ ]:
# Aşama 2 (güncel, docs §7): hepsi 20 epoch, oylama sqrt_inv, öteleme 0 (ref) — ~40 dk/koşu, toplam ~2,7 saat
# p2_ref: varsayılan | p2_dam0: tek örnekleme | p2_w1133: CE sınıf ağırlığı | p2_shift16: öteleme 16
RUNS = {
  "p2_ref":     "--post-shift 0  --dam-p 0.5",
  "p2_dam0":    "--post-shift 0  --dam-p 0.0",
  "p2_w1133":   "--post-shift 0  --dam-p 0.5 --cls-weights 1,1,3,3",
  "p2_shift16": "--post-shift 16 --dam-p 0.5",
}
for name, extra in RUNS.items():
    !python -m src.train_changeos --scenario S1 --holdout Malatya --epochs 20 {extra} \
        --cache-dir /kaggle/working/cache --workers 4 --max-minutes 60 --out /kaggle/working/runs/{name}
    # eğitim-içi kontrol (1 dk)
    !python -m src.train_changeos --scenario S1 --holdout Malatya --eval-only --eval-in-sample --ckpt /kaggle/working/runs/{name}/model_final.pt \
        --cache-dir /kaggle/working/cache --out /kaggle/working/runs/{name}_insample

In [ ]:
# Özet tablo (tüm koşular)
!python -m src.summarize_runs /kaggle/working/runs

Sonuçları (özet tablo + `timing.json`) olduğu gibi yapıştır; `notlar/adim8-durum.md` ve checklist oradan güncellenir.

## Aşama 3 — TANI: hasar neden öğrenilmiyor? (hizalama nedensellik testi; ~1 saat)
Aşama 2 sonucu: Malatya hasar-F1 ≈ 0 (hiçbir seçenek `p2_ref`i 0,07 geçmedi), eğitim-içi bile 0,19–0,30. Şüphe: sonrası görüntü etiketlere göre 14–45 px kaymış, hasar başı yanlış piksele bakıyor. Test: sonrası görüntüyü manifest kaymasıyla hizalayıp (ORACLE: kayma etiketten ölçülmüş, gerçek yöntem DEĞİL) aynı eğitimi tekrarla. Protokol: `docs/adim8_pilot_kaggle.md` §8.

In [ ]:
# (a) ÜCRETSİZ ön bakış: p2_ref'in TÜM eğitim karolarında (320) şehir ve kayma kademesine göre hasar-F1 (~4 dk)
!python -m src.train_changeos --scenario S1 --holdout Malatya --eval-only --eval-in-sample --in-sample-n 320 \
    --ckpt /kaggle/working/runs/p2_ref/model_final.pt --cache-dir /kaggle/working/cache --out /kaggle/working/runs/d_p2ref_insample320
import json
M = json.load(open("/kaggle/working/runs/d_p2ref_insample320/metrics.json"))["by"]
for k, v in M.items():
    if k.startswith(("group |", "group_tier |", "tier |")):
        print(f"{k:32s} karo {v['n_tiles']:4d} bina(sağlam/hasarlı/yıkık) {v['strict']['n_gt_by_class']} katı hasarF1 {v['strict']['damage_f1']:.3f} tol15 {v['tol15']['damage_f1']:.3f}")

In [ ]:
# (b) ORACLE hizalı eğitim (aynı ayarlar: 20 epoch, öteleme 0, dam-p 0.5, sqrt_inv) + hizalı eğitim-içi kontrol (~45 dk)
# önbellek ayrı dosyalara yazılır (_al.npy); hizasız önbellek bozulmaz
!python -m src.train_changeos --scenario S1 --holdout Malatya --epochs 20 --post-shift 0 --dam-p 0.5 --align-post oracle \
    --cache-dir /kaggle/working/cache --workers 4 --max-minutes 70 --out /kaggle/working/runs/p3_oracle
!python -m src.train_changeos --scenario S1 --holdout Malatya --eval-only --eval-in-sample --in-sample-n 320 --align-post oracle \
    --ckpt /kaggle/working/runs/p3_oracle/model_final.pt --cache-dir /kaggle/working/cache --out /kaggle/working/runs/p3_oracle_insample320
!python -m src.summarize_runs /kaggle/working/runs